Start with dask.

I'm running this on my laptop, but if you have more resources you can increase the n_workers (see more at https://docs.lsdb.io/en/latest/tutorials/dask_client.html)

In [1]:
import logging

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("distributed").setLevel(logging.WARNING)

In [2]:
# Make a dask client; set up dashboard for slacd

import os
import socket
from dask.distributed import Client

client = Client(
    n_workers=8,
    # threads_per_worker=2,
    memory_limit="40GB",
)

username = os.getenv("USER")
current_compute_node = socket.gethostname()
dashboard_port = client.scheduler_info()["services"]["dashboard"]

print(f"Run this in a local shell:")
print(
    f"  ssh -L 8787:{current_compute_node}:{dashboard_port} -J {username}@s3dflogin.slac.stanford.edu {username}@{current_compute_node}"
)
print(f"\nThen open: http://localhost:8787/status")


/sdf/group/rubin/sw/conda/envs/lsst-scipipe-13.0.0/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 11031 instead
  warnings.warn(


Run this in a local shell:
  ssh -L 8787:sdfiana030:11031 -J olynn@s3dflogin.slac.stanford.edu olynn@sdfiana030

Then open: http://localhost:8787/status


A sample dataframe (replace this with your sources table):

In [3]:
from lsdb import generate_data
from lsdb import ConeSearch

gen_cat = generate_data(10_000, 1, search_region=ConeSearch(5, 5, 1 * 3600))
df = gen_cat.compute()
df

ra       dec     id         a         b  \
0     5.529648  5.138661   2424  0.580665  1.904686   
1     5.150973  4.065236  72102  0.222001  1.086336   
...        ...       ...    ...       ...       ...   
9998  5.005753  4.965091  26132  0.024621  0.113132   
9999  5.571158  5.250487  35721  0.434244  1.147145   

                                                 nested  
0     [{t: 1.141218, flux: 21.841476, flux_error: 1....  
1     [{t: 19.577448, flux: 66.096478, flux_error: 1...  
...                                                 ...  
9998  [{t: 11.046422, flux: 65.908841, flux_error: 1...  
9999  [{t: 7.732554, flux: 32.536799, flux_error: 1....  

[10000 rows x 6 columns]

Use any catalog at [data.lsdb.io](data.lsdb.io):

In [4]:
import lsdb


gaia = lsdb.open_catalog("s3://stpubdata/gaia/gaia_dr3/public/hats")
desi = lsdb.open_catalog("s3://stpubdata/mast/public/desi/hats/desi_dr1_zcat")
tess = lsdb.open_catalog("s3://stpubdata/tess/public/hats/tic/")

In [5]:
gaia_x_df = lsdb.crossmatch(df, gaia)
gaia_x_df

/sdf/home/o/olynn/.local/lib/python3.13/site-packages/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


,ra_left,dec_left,id_left,a_left,b_left,nested_left,solution_id_gaia,designation_gaia,source_id_gaia,random_index_gaia,ref_epoch_gaia,ra_gaia,ra_error_gaia,dec_gaia,dec_error_gaia,parallax_gaia,parallax_error_gaia,parallax_over_error_gaia,pm_gaia,pmra_gaia,pmra_error_gaia,pmdec_gaia,pmdec_error_gaia,ra_dec_corr_gaia,ra_parallax_corr_gaia,ra_pmra_corr_gaia,ra_pmdec_corr_gaia,dec_parallax_corr_gaia,dec_pmra_corr_gaia,dec_pmdec_corr_gaia,parallax_pmra_corr_gaia,parallax_pmdec_corr_gaia,pmra_pmdec_corr_gaia,astrometric_n_obs_al_gaia,astrometric_n_obs_ac_gaia,astrometric_n_good_obs_al_gaia,astrometric_n_bad_obs_al_gaia,astrometric_gof_al_gaia,astrometric_chi2_al_gaia,astrometric_excess_noise_gaia,astrometric_excess_noise_sig_gaia,astrometric_params_solved_gaia,astrometric_primary_flag_gaia,nu_eff_used_in_astrometry_gaia,pseudocolour_gaia,pseudocolour_error_gaia,ra_pseudocolour_corr_gaia,dec_pseudocolour_corr_gaia,parallax_pseudocolour_corr_gaia,pmra_pseudocolour_corr_gaia,pmdec_pseudocolour_corr_gaia,astrometric_matched_transits_gaia,visibility_periods_used_gaia,astrometric_sigma5d_max_gaia,matched_transits_gaia,new_matched_transits_gaia,matched_transits_removed_gaia,ipd_gof_harmonic_amplitude_gaia,ipd_gof_harmonic_phase_gaia,ipd_frac_multi_peak_gaia,ipd_frac_odd_win_gaia,ruwe_gaia,scan_direction_strength_k1_gaia,scan_direction_strength_k2_gaia,scan_direction_strength_k3_gaia,scan_direction_strength_k4_gaia,scan_direction_mean_k1_gaia,scan_direction_mean_k2_gaia,scan_direction_mean_k3_gaia,scan_direction_mean_k4_gaia,duplicated_source_gaia,phot_g_n_obs_gaia,phot_g_mean_flux_gaia,phot_g_mean_flux_error_gaia,phot_g_mean_flux_over_error_gaia,phot_g_mean_mag_gaia,phot_bp_n_obs_gaia,phot_bp_mean_flux_gaia,phot_bp_mean_flux_error_gaia,phot_bp_mean_flux_over_error_gaia,phot_bp_mean_mag_gaia,phot_rp_n_obs_gaia,phot_rp_mean_flux_gaia,phot_rp_mean_flux_error_gaia,phot_rp_mean_flux_over_error_gaia,phot_rp_mean_mag_gaia,phot_bp_rp_excess_factor_gaia,phot_bp_n_contaminated_transits_gaia,phot_bp_n_blended_transits_gaia,phot_rp_n_contaminated_transits_gaia,phot_rp_n_blended_transits_gaia,phot_proc_mode_gaia,bp_rp_gaia,bp_g_gaia,g_rp_gaia,radial_velocity_gaia,radial_velocity_error_gaia,rv_method_used_gaia,rv_nb_transits_gaia,rv_nb_deblended_transits_gaia,rv_visibility_periods_used_gaia,rv_expected_sig_to_noise_gaia,rv_renormalised_gof_gaia,rv_chisq_pvalue_gaia,rv_time_duration_gaia,rv_amplitude_robust_gaia,rv_template_teff_gaia,rv_template_logg_gaia,rv_template_fe_h_gaia,rv_atm_param_origin_gaia,vbroad_gaia,vbroad_error_gaia,vbroad_nb_transits_gaia,grvs_mag_gaia,grvs_mag_error_gaia,grvs_mag_nb_transits_gaia,rvs_spec_sig_to_noise_gaia,phot_variable_flag_gaia,l_gaia,b_gaia,ecl_lon_gaia,ecl_lat_gaia,in_qso_candidates_gaia,in_galaxy_candidates_gaia,non_single_star_gaia,has_xp_continuous_gaia,has_xp_sampled_gaia,has_rvs_gaia,has_epoch_photometry_gaia,has_epoch_rv_gaia,has_mcmc_gspphot_gaia,has_mcmc_msc_gaia,in_andromeda_survey_gaia,classprob_dsc_combmod_quasar_gaia,classprob_dsc_combmod_galaxy_gaia,classprob_dsc_combmod_star_gaia,teff_gspphot_gaia,teff_gspphot_lower_gaia,teff_gspphot_upper_gaia,logg_gspphot_gaia,logg_gspphot_lower_gaia,logg_gspphot_upper_gaia,mh_gspphot_gaia,mh_gspphot_lower_gaia,mh_gspphot_upper_gaia,distance_gspphot_gaia,distance_gspphot_lower_gaia,distance_gspphot_upper_gaia,azero_gspphot_gaia,azero_gspphot_lower_gaia,azero_gspphot_upper_gaia,ag_gspphot_gaia,ag_gspphot_lower_gaia,ag_gspphot_upper_gaia,ebpminrp_gspphot_gaia,ebpminrp_gspphot_lower_gaia,ebpminrp_gspphot_upper_gaia,libname_gspphot_gaia,_dist_arcsec
npartitions=2,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 2, Pixel: 70",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],"nested<t: [double], flux: [double], flux_error...",int64[pyarrow],string[pyarrow],int64[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],float[pyar

In [6]:
gaia_x_df_computed = gaia_x_df.compute()
gaia_x_df_computed

Computing Catalog:   0%|          | 0/2 [00:00<?, ?it/s]

ra_left  dec_left  id_left    a_left    b_left  \
_healpix_29                                                            
1274518193337336624  5.572965  4.690696    23507  0.733639  1.520739   
1274447744302717747  4.952798  4.121256    36769  0.133234  0.237702   
1370588918101638804  4.976591  4.713736    35428  0.730786   1.13968   
1373741591619967780   4.78472  5.542743    74263  0.603334  1.883052   
1370770658077256863  4.332706   5.60502    84811  0.893689  0.760562   
1373618697949269531  5.871569  5.275921    53963  0.460093   0.95011   
1370611692325474308  4.760143  4.657191    42355   0.54493  0.291811   
1370627824970835306   4.78124  4.958612    89504  0.867546  1.351432   
1370753639034714739    4.1106  5.343628    21745  0.110067  1.934293   
1370580068207385417  5.145177  4.552219    84627  0.208912  0.060409   
1370768263915747581  4.169384  5.545794    42446  0.977244  1.780052   

                                                           nested_left  \
_healpix_29                                                              
1274518193337336624  [{t: 3.409178, flux: 37.71742, flux_error: 1.0...   
1274447744302717747  [{t: 4.303294, flux: 82.253538, flux_error: 1....   
1370588918101638804  [{t: 12.699027, flux: 95.69947, flux_error: 1....   
1373741591619967780  [{t: 3.281648, flux: 16.711521, flux_error: 1....   
1370770658077256863  [{t: 3.122818, flux: 19.910746, flux_error: 1....   
1373618697949269531  [{t: 13.395052, flux: 76.042471, flux_error: 1...   
1370611692325474308  [{t: 0.540427, flux: 95.637754, flux_error: 1....   
1370627824970835306  [{t: 11.650692, flux: 94.08544, flux_error: 1....   
1370753639034714739  [{t: 14.430049, flux: 13.057236, flux_error: 1...   
1370580068207385417  [{t: 11.906497, flux: 30.042486, flux_error: 1...   
1370768263915747581  [{t: 7.058015, flux: 56.045911, flux_error: 1....   

                        solution_id_gaia              designation_gaia  \
_healpix_29                                                              
1274518193337336624  1636148068921376768  Gaia DR3 2549036358299792768   
1274447744302717747  1636148068921376768  Gaia DR3 2548895483372444032   
1370588918101638804  1636148068921376768  Gaia DR3 2741177806255797504   
1373741591619967780  1636148068921376768  Gaia DR3 2747483157549231744   
1370770658077256863  1636148068921376768  Gaia DR3 2741541293632129920   
1373618697949269531  1636148068921376768  Gaia DR3 2747237382339226496   
1370611692325474308  1636148068921376768  Gaia DR3 2741223362973165952   
1370627824970835306  1636148068921376768  Gaia DR3 2741255626768395008   
1370753639034714739  1636148068921376768  Gaia DR3 2741507277491270272   
1370580068207385417  1636148068921376768  Gaia DR3 2741160106696452352   
1370768263915747581  1636148068921376768  Gaia DR3 2741536517629766656   

                          source_id_gaia  random_index_gaia  ...  \
_healpix_29                                                  ...   
1274518193337336624  2549036358299792768         1684716544  ...   
1274447744302717747  2548895483372444032         1387664650  ...   
1370588918101638804  2741177806255797504          909281497  ...   
1373741591619967780  2747483157549231744          278756396  ...   
1370770658077256863  2741541293632129920           38537972  ...   
1373618697949269531  2747237382339226496          830709379  ...   
1370611692325474308  2741223362973165952          285109328  ...   
1370627824970835306  2741255626768395008         1405369769  ...   
1370753639034714739  2741507277491270272          621931126  ...   
1370580068207385417  2741160106696452352         1088246784  ...   
1370768263915747581  2741536517629766656          690528055  ...   

                     azero_gspphot_lower_gaia  azero_gspphot_upper_gaia  \
_healpix_29                                                               
1274518193337336624                      <NA>                      <NA>   
1274447744302717747                      <NA>    

In [ ]:
desi_x_df = lsdb.crossmatch(df, gaia)
desi_x_df_computed = desi_x_df.compute()
desi_x_df_computed